# TMT data treatment

This code will work with the TMT data provided by the FGCZ.

Initial working file for each experiment (hTERT_HME1_1, hTERT_HME1_2, HEK293T_1) is the raw_abundances_matrix

**First step is to rename the files to have a unified labeling system** -> (CellLine)_ (Data:Type)_ (Treatment)_ (TimePoint)_ (Replicate)

All the data transformation and statistics I am going to do starting from that initial "raw_data" file.



In [1]:
from src.transformations import *  # all necessary pakages are imported within src.transformations

# hme1_1 = pd.read_csv("../../data/hme1_1_raw_sample.tsv", sep="\t")
# hme1_2 = pd.read_csv("../../data/hme1_2_raw_sample.tsv", sep="\t")
# hek_1 = pd.read_csv("../../data/hek_1_raw_sample.tsv", sep="\t")
#
hme1_1 = pd.read_csv("../../Experiment/hme1_1/Data/All/20260427_hTERT_HME1_1.tsv", sep="\t")
hme1_2 = pd.read_csv("../../Experiment/hme1_2/Data/All/20260427_hTERT_HEM1_2.tsv", sep="\t")
hek_1 = pd.read_csv("../../Experiment/hek_1/Data/All/20260417_HEK293T_raw_selection.tsv", sep="\t")

## Data transformations

Run the full transformation pipeline on each dataset:
- n:reps  # important to double check this number of reps identified for the LFQ dataset
- raw:mean / raw:median / raw:sd / raw:cv
- log2:abs (zeros treated as NaN)
- log2:mean / log2:median / log2:sd
- log2:FC (fold change vs. starve)
- log2:scaled (max-normalised fold change, amplitude between -1 and 1)
- log2:zscore (per-site z-score of the temporal profile, computed per condition per cell line across the time series; removes amplitude, keeps shape)
- log2:pvalue (Welch t-test vs. starve)
- log2:FDR (Benjamini-Hochberg correction per treatment × timepoint)
- log2:adjustedFDR (-log10 transformation of the FDR, also called adjusted FDR or adjusted p-value)


In [2]:
hme1_1_transformed = run_all_transformations(hme1_1, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hme1_2_transformed = run_all_transformations(hme1_2, cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])
hek_1_transformed  = run_all_transformations(hek_1,  cell_lines=["WT"], data_type="raw:abs", conditions=['_EGF_', '_INS_', '_EGFnINS_'])

print(f"hme1_1: {hme1_1.shape} -> {hme1_1_transformed.shape}")
print(f"hme1_2: {hme1_2.shape} -> {hme1_2_transformed.shape}")
print(f"hek_1:  {hek_1.shape}  -> {hek_1_transformed.shape}")

[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 349 columns.

All cell lines processed. Total columns: 90 -> 439
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 349 columns.

All cell lines processed. Total columns: 106 -> 455
[WT] Found 3 conditions, 21 (treatment, timepoint) groups.
[WT] Done. Added 349 columns.

All cell lines processed. Total columns: 106 -> 455
hme1_1: (35002, 90) -> (35002, 439)
hme1_2: (50002, 106) -> (50002, 455)
hek_1:  (36567, 106)  -> (36567, 455)


In [3]:
# Quick check of the new per-site z-score columns (log2:zscore) produced by run_all_transformations.
# Each condition's time series is standardised independently per site: mean ~ 0, std ~ 1 across timepoints.
from src.column_spec import ColumnSpec

_z_egf = ColumnSpec.select(hme1_2_transformed, cell_lines=["WT"], data_type="log2:zscore", conditions=["_EGF_"])
print("EGF z-score columns:", _z_egf)
hme1_2_transformed[_z_egf].head()

EGF z-score columns: ['WT_log2:zscore_EGF_full', 'WT_log2:zscore_EGF_starve', 'WT_log2:zscore_EGF_2', 'WT_log2:zscore_EGF_5', 'WT_log2:zscore_EGF_10', 'WT_log2:zscore_EGF_15', 'WT_log2:zscore_EGF_90']


,WT_log2:zscore_EGF_full,WT_log2:zscore_EGF_starve,WT_log2:zscore_EGF_2,WT_log2:zscore_EGF_5,WT_log2:zscore_EGF_10,WT_log2:zscore_EGF_15,WT_log2:zscore_EGF_90
0,1.719098,-0.366660,-1.061852,-0.433478,0.966686,-1.224618,0.400823
1,2.250126,-0.865472,-0.914893,-0.237049,0.310669,-0.430627,-0.112755
2,-1.291940,-0.102210,-0.074980,1.774072,-0.179459,-1.094103,0.968620
3,-0.312939,1.921967,-1.493313,0.544728,-0.107268,-0.784965,0.231790
4,-1.264736,-0.489006,-0.733860,1.633720,1.217400,-0.632087,0.268568


In [4]:
# Save transformed datasets
# hek_1_transformed.to_csv("../../Experiment/hek_1/Data/Processed/20260714_HEK293T_processed.tsv", sep="\t", index=False)
# hme1_1_transformed.to_csv("../../Experiment/hme1_1/Data/Processed/20260714_hTERT_HEM1_1_processed.tsv", sep="\t", index=False)
# hme1_2_transformed.to_csv("../../Experiment/hme1_2/Data/Processed/20260714_hTERT_HEM1_2_processed.tsv", sep="\t", index=False)

## Merging external data

PhosphoSite Plus dataset

In [5]:
functional_score_df = pd.read_csv("../../External_Data/Metadata/PhosphoSitePlus.tsv", sep="\t")
regulatory_sites = pd.read_csv("../../External_Data/Metadata/Phosphosite/Regulatory_sites.tsv", sep="\t")

print(f"functional_score_df columns: {functional_score_df.columns}")
print(f"regulatory_sites columns: {regulatory_sites.columns}")

functional_score_df columns: Index(['protein_Id', 'protein_name', 'prot_seq_position', 'aa', 'site',
       'functional_score', 'ms_lit', 'ERK_motif', 'ERK_ext_motif'],
      dtype='object')
regulatory_sites columns: Index(['GENE', 'protein_name', 'information', 'protein_Id', 'GENE_ID',
       'HU_CHR_LOC', 'ORGANISM', 'MOD_RSD', 'SITE_GRP_ID', 'SITE_+/-7_AA',
       'DOMAIN', 'ON_FUNCTION', 'ON_PROCESS', 'ON_PROT_INTERACT',
       'ON_OTHER_INTERACT', 'PMIDs', 'LT_LIT', 'MS_LIT', 'MS_CST', 'NOTES'],
      dtype='object')


In [10]:
df_with_extra_info = merge_functional_score(df = hek_1_transformed,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position")
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ERK_motif"])
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = regulatory_sites,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ORGANISM", "ON_FUNCTION", "ON_PROCESS", "ON_PROT_INTERACT", 'ON_OTHER_INTERACT'],
                                            regulatory_sites= True)
df_with_extra_info

,protein_Id,nr_peptides,description,protein_name,protein_length,nr_tryptic_peptides,peptide_index,peptide_seq,SequenceWindow,Start,...,WT_log2:adjustedFDR_EGFnINS_10,WT_log2:adjustedFDR_EGFnINS_15,WT_log2:adjustedFDR_EGFnINS_90,functional_score,ERK_motif,ORGANISM,ON_FUNCTION,ON_PROCESS,ON_PROT_INTERACT,ON_OTHER_INTERACT
0,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_0,SGSGNFGGGR,SSSQRGR.SGSGNFGGGR.GGGFGGN,197,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_197_199_1_1_S199,SGsGNFGGGR,SSSQRGR.SGsGNFGGGR.GGGFGGN,197,...,0.088086,0.032852,0.558304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_237_271_1_0,GGGGYGGSGDGYNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,GGFGGSR.GGGGYGGSGDGYNGFGNDGSNFGGGGSYNDFGNYNNQS...,233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_237_271_1_1_Y244,GGGGYGGSGDGyNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMK,GGFGGSR.GGGGYGGSGDGyNGFGNDGSNFGGGGSYNDFGNYNNQS...,233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0A2R8Y4L2,13,Heterogeneous nuclear ribonucleoprotein A1-lik...,HNRNPA1L3,320.0,18.0,A0A2R8Y4L2_305_316_1_0,NQGGYGGSSSSSSYGSGR;NQGGYGGSSSSSSYGSGRRF,QYFAKPR.NQGGYGGSSSSSSYGSGR.RF;QYFAKPR.NQGGYGGS...,301,...,0.850243,0.076864,0.215380,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36562,Q9Y561,1,Low-density lipoprotein receptor-related prote...,LRP12,859.0,40.0,Q9Y561_646_657_1_1_T655,SLFSVESDDtDTENERR,ERNHTHR.SLFSVESDDtDTENERR.DMAGASG,646,...,NaN,NaN,NaN,0.450637,False,NaN,NaN,NaN,NaN,NaN
36563,Q9Y5U4,1,Insulin-induced gene 2 protein OS=Homo sapiens...,INSIG2,225.0,9.0,Q9Y5U4_6_8_1_1_S8,AEGETEsPGPKK,.AEGETEsPGPKK.CGPYISS,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36564,Q9Y5Y9,1,Sodium channel protein type 10 subunit alpha O...,SCN10A,1956.0,82.0,Q9Y5Y9_954_971_3_3_S954S955S956,LPLsssKAENHIAANTARGSSGGLQAPR,EPEIVVK.LPLsssKAENHIAANTARGSSGGLQAPR.GPRDEHS,951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36565,Q9Y676,1,Small ribosomal subunit protein mS40 OS=Homo s...,MRPS18B,258.0,13.0,Q9Y676_38_51_1_1_S38,APsEEDSLSSVPISPYKDEPWK,IQTICTK.APsEEDSLSSVPISPYKDEPWK.YIESEEY,36,...,NaN,NaN,NaN,0.332708,False,NaN,NaN,NaN,NaN,NaN


In [11]:
# df_with_extra_info.to_csv("../../Experiment/hek_1/Data/Processed/20260714_HEK293T_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_1/Data/Processed/20260714_hTERT_HEM1_1_processed_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_2/Data/Processed/20260714_hTERT_HEM1_2_processed_phPlus.tsv", sep="\t", index=False)

In [ ]:
# df_with_extra_info.loc[df_with_extra_info["protein_name"] == "EGFR"]